In [1]:
import bw2data, bw2io, bw2calc
from bw_timex import TimexLCA
from bw_temporalis import TemporalDistribution, easy_timedelta_distribution
import numpy as np
from datetime import datetime
import os
import re
import pandas as pd
import numpy as np
import pickle

In [2]:
import sys
sys.path.append('../../utils/') 
from elec_builder import *

In [3]:
# activate the bw project
bw2data.projects.set_current("ei311")
#for db in bw2data.databases:
#    print(db, len(bw2data.Database(db)))

### 1. building wind_foreground 
- China,  nner mongolia: https://pubmed.ncbi.nlm.nih.gov/36123557/ (Fifty-six percent of the potential is located in Inner Mongolia, Xinjiang, and Gansu provinces)
- the U.S. , Texas (US-TRE: Texas Regional Entity) was chose for two reasons1.  it has high elec demand, 2. it has huge wind potential :  https://docs.nrel.gov/docs/fy11osti/51555.pdf 

## (p)GWP20 calc

In [4]:
wind_db = bw2data.Database("elec_wind_foreground")
len(wind_db)
list(wind_db) 

['market for electricity, wind, high voltage, US-TRE, SSP5-H, 2030' (kWh, US-TRE, None),
 'market for electricity, wind, high voltage, US-TRE, SSP5-H, 2050' (kWh, US-TRE, None),
 'market for electricity, wind, high voltage, US-TRE, SSP2-M, 2030' (kWh, US-TRE, None),
 'market for electricity, wind, high voltage, CN-NM, SSP5-H, 2030' (kWh, CN-NM, None),
 'market for electricity, wind, high voltage, US-TRE, SSP5-H, 2040' (kWh, US-TRE, None),
 'market for electricity, wind, high voltage, CN-NM, SSP1-VLLO, 2050' (kWh, CN-NM, None),
 'market for electricity, wind, high voltage, CN-NM, SSP2-M, 2050' (kWh, CN-NM, None),
 'market for electricity, wind, high voltage, US-TRE, SSP1-VLLO, 2050' (kWh, US-TRE, None),
 'market for electricity, wind, high voltage, US-TRE, SSP2-M, 2050' (kWh, US-TRE, None),
 'market for electricity, wind, high voltage, US-TRE, SSP2-M, 2040' (kWh, US-TRE, None),
 'market for electricity, wind, high voltage, CN-NM, SSP5-H, 2040' (kWh, CN-NM, None),
 'market for electricit

In [5]:
# only calc 2030 and 2050 act: 
wind_db_2030_50 = [
    act for act in wind_db 
    if ( "2050" in str(act.get('name', ''))  or ("2030" in str(act.get('name', ''))) )
]

wind_db_2030_50

['market for electricity, wind, high voltage, CN-NM, SSP5-H, 2030' (kWh, CN-NM, None),
 'market for electricity, wind, high voltage, CN-NM, SSP2-M, 2050' (kWh, CN-NM, None),
 'market for electricity, wind, high voltage, CN-NM, SSP2-M, 2030' (kWh, CN-NM, None),
 'market for electricity, wind, high voltage, CN-NM, SSP1-VLLO, 2030' (kWh, CN-NM, None),
 'market for electricity, wind, high voltage, CN-NM, SSP1-VLLO, 2050' (kWh, CN-NM, None),
 'market for electricity, wind, high voltage, US-TRE, SSP2-M, 2050' (kWh, US-TRE, None),
 'market for electricity, wind, high voltage, US-TRE, SSP2-M, 2030' (kWh, US-TRE, None),
 'market for electricity, wind, high voltage, US-TRE, SSP5-H, 2030' (kWh, US-TRE, None),
 'market for electricity, wind, high voltage, CN-NM, SSP5-H, 2050' (kWh, CN-NM, None),
 'market for electricity, wind, high voltage, US-TRE, SSP1-VLLO, 2050' (kWh, US-TRE, None),
 'market for electricity, wind, high voltage, US-TRE, SSP1-VLLO, 2030' (kWh, US-TRE, None),
 'market for electric

In [6]:
rows = []   # collect results for all acts
act_list = list(wind_db_2030_50)   

for act in act_list: 
    print(act)

    name = act.get("name")
    name_parts = [p.strip() for p in name.split(",")]
    # run static LCI + premise_GWP vs. pGWP20 first: 
    pgwp_fixedco2 = find_dpGWP100_method(name_parts[-2], int(name_parts[-1]), 
                                         method_prefix = 'Climate Change prospective GWP20', 
                                         method_suffix = "pGWP20 - fixed-AGWPCO2")
    pgwp_dpco2 = find_dpGWP100_method(name_parts[-2], int(name_parts[-1]), 
                                      method_prefix = 'Climate Change prospective GWP20', 
                                      method_suffix = "pGWP20 - dp-AGWPCO2")
    gwp =  ("ecoinvent-3.11", "IPCC 2021", 'climate change: total (incl. biogenic CO2 new)',  'global warming potential (GWP20)') 

    lca_gwp = bw2calc.lca.LCA({act: 1}, method=gwp)
    lca_gwp.lci(); lca_gwp.lcia()
    score_gwp = float(lca_gwp.score)

    lca_fixed = bw2calc.lca.LCA({act: 1}, method=pgwp_fixedco2)
    lca_fixed.lci(); lca_fixed.lcia()
    score_fixedco2 = float(lca_fixed.score)

    lca_dp = bw2calc.lca.LCA({act: 1}, method=pgwp_dpco2)
    lca_dp.lci(); lca_dp.lcia()
    score_dpco2 = float(lca_dp.score)

    print(score_gwp, score_fixedco2, score_dpco2) 

    # ---- store results for this activity ----
    rows.append({
        "Activity": name,                      # index value later
        "gwp20": score_gwp,
        "pGWP20_fixedCO2": score_fixedco2,
        "pGWP20_dpCO2": score_dpco2,
    })

'market for electricity, wind, high voltage, CN-NM, SSP5-H, 2030' (kWh, CN-NM, None)
0.04291469258076569 0.047800865434146564 0.043580849201912145
'market for electricity, wind, high voltage, CN-NM, SSP2-M, 2050' (kWh, CN-NM, None)
0.03467615139036944 0.03613505816291967 0.036289693427928996
'market for electricity, wind, high voltage, CN-NM, SSP2-M, 2030' (kWh, CN-NM, None)
0.04183674909570945 0.04765129868256729 0.04275591218132756
'market for electricity, wind, high voltage, CN-NM, SSP1-VLLO, 2030' (kWh, CN-NM, None)
0.04097710970076182 0.0480069547914517 0.04213135634687005
'market for electricity, wind, high voltage, CN-NM, SSP1-VLLO, 2050' (kWh, CN-NM, None)
0.020379066541318117 0.024405635509240763 0.021572955579318064
'market for electricity, wind, high voltage, US-TRE, SSP2-M, 2050' (kWh, US-TRE, None)
0.01939114797640654 0.020206978914293454 0.020293452043113606
'market for electricity, wind, high voltage, US-TRE, SSP2-M, 2030' (kWh, US-TRE, None)
0.023395404623668722 0.02664

In [7]:
df_scores = pd.DataFrame(rows)
df_scores = df_scores.set_index("Activity")

df2 = df_scores.sort_values(by=['Activity'])
df2 

,gwp20,pGWP20_fixedCO2,pGWP20_dpCO2
Activity,,,
"market for electricity, wind, high voltage, CN-NM, SSP1-VLLO, 2030",0.040977,0.048007,0.042131
"market for electricity, wind, high voltage, CN-NM, SSP1-VLLO, 2050",0.020379,0.024406,0.021573
"market for electricity, wind, high voltage, CN-NM, SSP2-M, 2030",0.041837,0.047651,0.042756
"market for electricity, wind, high voltage, CN-NM, SSP2-M, 2050",0.034676,0.036135,0.036290
"market for electricity, wind, high voltage, CN-NM, SSP5-H, 2030",0.042915,0.047801,0.043581
"market for electricity, wind, high voltage, CN-NM, SSP5-H, 2050",0.038172,0.035892,0.039536
"market for electricity, wind, high voltage, US-TRE, SSP1-VLLO, 2030",0.022915,0.026846,0.023560
"market for electricity, wind, high voltage, US-TRE, SSP1-VLLO, 2050",0.011396,0.013648,0.012064
"market for electricity, wind, high voltage, US-TRE, SSP2-M, 2030",0.023395,0.026647,0.023909


In [8]:
df2.to_excel("../dp-LCI_output/staticLCI_(p)GWP20/wind_CN_US_staticLCI_threeGWP20.xlsx")

## 2. WIP building dpLCI

In [ ]:
wind_db = bw2data.Database("elec_wind_foreground")

# only calc 2030 and 2050 act: 
wind_db_2030_50 = [
    act for act in wind_db 
    if ( "2050" in str(act.get('name', ''))  or ("2030" in str(act.get('name', ''))) )
]

wind_db_2030_50

In [10]:
database_dates = {
    'ei_cutoff_3.11_image_SSP1-VLLO_2030 2025-11-24': datetime.strptime("2030", "%Y"),
    'ei_cutoff_3.11_image_SSP1-VLLO_2040 2025-11-22': datetime.strptime("2040", "%Y"),
    'ei_cutoff_3.11_image_SSP1-VLLO_2050 2025-11-22': datetime.strptime("2050", "%Y"),

    'ei_cutoff_3.11_image_SSP2-M_2030 2025-11-22': datetime.strptime("2030", "%Y"),
    'ei_cutoff_3.11_image_SSP2-M_2040 2025-11-22': datetime.strptime("2040", "%Y"),
    'ei_cutoff_3.11_image_SSP2-M_2050 2025-11-22': datetime.strptime("2050", "%Y"),

    'ei_cutoff_3.11_image_SSP5-H_2030 2025-11-22': datetime.strptime("2030", "%Y"),
    'ei_cutoff_3.11_image_SSP5-H_2040 2025-11-22': datetime.strptime("2040", "%Y"),
    'ei_cutoff_3.11_image_SSP5-H_2050 2025-11-22': datetime.strptime("2050", "%Y"),
    
    "elec_wind_foreground": "dynamic", # flag databases that should be temporally distributed with "dynamic"
}


In [11]:
dp_results = {}

for act in list(wind_db_2030_50): 
    print(act)
 
    #### dynamic LCI:: 
    assign_td_from_foreground_db( 
        select_act = act,
        elec_td_year=10,
        resolution="Y",
        kind="uniform",
        fg_db_name="elec_wind_foreground",
        verbose = True
     )

    tlca = run_dp_timex_lca(foreground_act = act ,   
                    pathway = None,
                    year = None,
                    method = None,
                    database_dates = database_dates, #None not working, has to incl. all 9 background DB ... 
                    temporal_grouping="year", 
                    method_prefix = "Climate Change prospective GWP100",
                    method_suffix = "pGWP100 - fixed-AGWPCO2", 
                    fg_db_name = 'elec_wind_foreground'
     )

    tlca.lci()
    tlca.dynamic_inventory.shape

    # dyn-foreground LCI + static background LCI + pGWP100
    lca_0 = tlca.base_score 
    print(lca_0)
    
    # dpLCI (dyn-foreground LCI + dyn background LCI) +  pGWP100
    tlca.static_lcia()
    lca_1 = tlca.static_score
    print(lca_1)

    act_name = act.get("name")
    dp_results[act_name] = {
        "dyn FG LCI static BG p-LCI, pGWP100-fixedCO2": float(lca_0),   # static BG LCI + dyn FG LCI
        "full dp-LCI, pGWP100-fixedCO2": float(lca_1),      # dyn BG LCI + dyn FG LCI
    }

    
    #### now save all dyLCI flows to pandas 
    df = tlca.dynamic_inventory_df

    ##### important to convert flow and act as str to excel 
    df["flow"] = df["flow"].astype(str)
    df["activity"] = df["activity"].astype(str)

    ### export df to dp-LCI_output folder, using the act name as the excel name 
    out_dir = "dp-LCI_output/wind_dpLCI"
    os.makedirs(out_dir, exist_ok=True)    
    raw_name = act["name"]
    safe_name = re.sub(r"[^A-Za-z0-9_\-()]+", "_", raw_name)   # replace spaces/special chars
    
    excel_path = os.path.join(out_dir, f"{safe_name}.xlsx")
    
    df.to_excel(excel_path, index=False)    
    print(f"✔ Exported dynamic inventory DF for '{raw_name}' → {excel_path}")





import pickle

out_dir = "dp-LCI_output/wind_dpLCI"
os.makedirs(out_dir, exist_ok=True)

pickle_path = os.path.join(out_dir, "wind_dp_results_allMY.pkl")

with open(pickle_path, "wb") as f:
    pickle.dump(dp_results, f)

print(f"✔ Saved dp_results dictionary → {pickle_path}")

2025-12-07 00:35:05.165 | INFO     | bw_timex.timex_lca:__init__:114 - Initializing TimexLCA object...
2025-12-07 00:35:05.167 | INFO     | bw_timex.timex_lca:__init__:136 - Collecting node infos...


'market for electricity, wind, high voltage, CN-NM, SSP5-H, 2040' (kWh, CN-NM, None)
TD applied to 'market for electricity, wind, high voltage, CN-NM, SSP5-H, 2040' → Exchange: 1 kilowatt hour 'electricity production, wind, >3MW turbine, onshore' (kilowatt hour, CN-NM, None) to 'market for electricity, wind, high voltage, CN-NM, SSP5-H, 2040' (kWh, CN-NM, None)>
for the activity 'market for electricity, wind, high voltage, CN-NM, SSP5-H, 2040' (kWh, CN-NM, None),it's under SSP-SSP5-H, year-2040 
 we'll use LCIA ('Climate Change prospective GWP100', 'SSP585', 'MY2040', 'pGWP100 - fixed-AGWPCO2')  with background database.datetime = {'ei_cutoff_3.11_image_SSP1-VLLO_2030 2025-11-24': datetime.datetime(2030, 1, 1, 0, 0), 'ei_cutoff_3.11_image_SSP1-VLLO_2040 2025-11-22': datetime.datetime(2040, 1, 1, 0, 0), 'ei_cutoff_3.11_image_SSP1-VLLO_2050 2025-11-22': datetime.datetime(2050, 1, 1, 0, 0), 'ei_cutoff_3.11_image_SSP2-M_2030 2025-11-22': datetime.datetime(2030, 1, 1, 0, 0), 'ei_cutoff_3.11

2025-12-07 00:38:42.097 | INFO     | bw_timex.timex_lca:build_timeline:216 - No edge filter function provided. Skipping all edges in background databases.
2025-12-07 00:40:11.536 | INFO     | bw_timex.timex_lca:build_timeline:232 - Calculating base LCA...
2025-12-07 00:40:22.183 | INFO     | bw_timex.timex_lca:build_timeline:242 - Creating activity time mapping...
2025-12-07 00:40:24.782 | INFO     | bw_timex.timeline_builder:__init__:99 - Traversing supply chain graph...


Starting graph traversal


2025-12-07 00:40:26.363 | INFO     | bw_timex.timeline_builder:build_timeline:142 - Building timeline...
2025-12-07 00:40:26.492 | INFO     | bw_timex.timeline_builder:get_weights_for_interpolation_between_nearest_years:522 - Reference date 2026-01-01 00:00:00 is lower than all provided dates. Data will be taken from the closest higher year.
2025-12-07 00:40:26.494 | INFO     | bw_timex.timeline_builder:get_weights_for_interpolation_between_nearest_years:522 - Reference date 2026-01-01 00:00:00 is lower than all provided dates. Data will be taken from the closest higher year.
2025-12-07 00:40:26.494 | INFO     | bw_timex.timeline_builder:get_weights_for_interpolation_between_nearest_years:522 - Reference date 2027-01-01 00:00:00 is lower than all provided dates. Data will be taken from the closest higher year.
2025-12-07 00:40:26.495 | INFO     | bw_timex.timeline_builder:get_weights_for_interpolation_between_nearest_years:522 - Reference date 2028-01-01 00:00:00 is lower than all prov

...

Starting graph traversal


2025-12-07 04:08:47.556 | INFO     | bw_timex.timeline_builder:build_timeline:142 - Building timeline...


Calculation count: 1


2025-12-07 04:08:47.707 | INFO     | bw_timex.timeline_builder:get_weights_for_interpolation_between_nearest_years:522 - Reference date 2026-01-01 00:00:00 is lower than all provided dates. Data will be taken from the closest higher year.
2025-12-07 04:08:47.709 | INFO     | bw_timex.timeline_builder:get_weights_for_interpolation_between_nearest_years:522 - Reference date 2026-01-01 00:00:00 is lower than all provided dates. Data will be taken from the closest higher year.
2025-12-07 04:08:47.712 | INFO     | bw_timex.timeline_builder:get_weights_for_interpolation_between_nearest_years:522 - Reference date 2027-01-01 00:00:00 is lower than all provided dates. Data will be taken from the closest higher year.
2025-12-07 04:08:47.714 | INFO     | bw_timex.timeline_builder:get_weights_for_interpolation_between_nearest_years:522 - Reference date 2028-01-01 00:00:00 is lower than all provided dates. Data will be taken from the closest higher year.
2025-12-07 04:08:47.716 | INFO     | bw_time

0.04152023302127886
0.04316385924257156
✔ Exported dynamic inventory DF for 'market for electricity, wind, high voltage, CN-NM, SSP1-VLLO, 2030' → dp-LCI_output/wind_dpLCI/market_for_electricity_wind_high_voltage_CN-NM_SSP1-VLLO_2030.xlsx
✔ Saved dp_results dictionary → dp-LCI_output/wind_dpLCI/wind_dp_results_allMY.pkl
